## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

We're chunking here because it's difficult to find relevant info from one single vector representing the entire doc

Encoder models turns text into vector embeddings
which are then stored in vector databases like chroma 

Popular Vectorstores:

Open-Source: Chroma, Qdrant, FAISS (in-memory)

Paid & scalable: Pinecone, Weaviate, ...

Mainstream databases (Postgres, Mongo, Elastic, ... )

Popular Encoders:
OpenAl text embedding
Gemini embedding
Hugging Face all-MiniLM-L6-v2

### PART A: Divide our documents into chunks

In [ ]:

import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [ ]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


In [ ]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

```files = glob.glob(knowledge_base_path, recursive=True)```

a list of filepaths, both in top folder and subdirectories using recursive=True

```for file_path in files: with... ```
open the files one by one and read all the content 
store in entire_knowledge_base with two newlines to seperate each file

```print(f"{n}")     # 1000000```
```print(f"{n:,}")   # 1,000,000```

The : introduces formatting rules.

The , means:

Add commas as thousands separators.



In [ ]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

```encoding_for_model```
returns the tokenizer for the specific model

```encode()```
converts the string into a list of tokens

In [ ]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

```doc_type = os.path.basename(folder)```
just the filename "products" or "company" - becomes metadata

we get the folder say knowledge-base/company
**/*.md relative to knowledge-base/company:

```knowledge-base/company/file1.md```

```knowledge-base/company/subfolder/nested/file3.md```

```DirectoryLoader``` is a LangChain loader that:

Recursively finds files matching **/*.md inside the folder

Uses TextLoader to read them

loader knows how to load all md files

.load() returns a list of Document objects 

```doc.page_content  # the text of the file```

```doc.metadata      # a dictionary of metadata```

In [ ]:
documents[1]

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Chunk 1: characters 0–1000

Chunk 2: characters 800–1800

In [ ]:
chunks[100]

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

In [ ]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

```Chroma(persist_directory=db_name, embedding_function=embeddings)```
creates a chroma database object pointing to folder db_name

```persist_direcory=db_name``` - the folder to look in instead of reloading everytime if no persist_directory

lost at the end of program, with it saves on disk in that folder

```embedding_function=embeddings```
specifies which embedding model to use if adding docs later

```.delete_collection()``` deletes entire collection of vectors to start fresh 

In [ ]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

```collection.get(limit=1, include=["embeddings"])```
returns a dictionary

```python
{
    "ids": ["abc123"],
    "documents": ["Some text"],
    "metadatas": [{}],
    "embeddings": [
        [0.0123, -0.4567, 0.9981, ...]
    ]
}
```
since we use include=["embeddings] we get

```python
{
    "embeddings": [
        [0.0123, -0.4567, 0.9981, ...]
    ]
}
```

embeddings[0] from the dictionary returned 

give me the values stored under key embeddings 

and give me first element of that list


```python
[0.0123, -0.4567, 0.9981, ...]
```


### Part C: Visualize!

In [ ]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

```python
doc_types = [metadata['doc_type'] for meta_data in meta_datas]
```

Loops over each metadata dictionary and pulls out the 'doc_type' field

Example: ['products', 'contracts', 'employees', 'company']

```python
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]
```

assign a colour to each document type

t is a single document type for example 'contracts'. index(t) searches for the value in the list and returns 

it's postition

```python
['products', 'employees', 'contracts', 'company'].index('contracts') # returns 2
```

'contracts' → index 2 → color 'red'.

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

```go.Figure(data=[...])``` -> creates a plotly figure

```data=[...]``` things to plot here it's just one trace go.Scatter(...)

The square brackets [...] make it a list, even if there’s only one trace. Plotly always expects a list for data.

```go.Scatter(...)``` creates a scatter plot 

```python
x=reduced_vectors[:, 0]
```
take all rows, first column 

(num_docs, 2) looks like this 

```python
reduced_vectors =
[[ 1.2,  3.4],   # Document 1 → x=1.2, y=3.4
 [ 5.6,  7.8],   # Document 2 → x=5.6, y=7.8
 [ 0.1, -2.3]]   # Document 3 → x=0.1, y=-2.3
```
mode='markers' only draw points

```python
text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)]
```
zip(doc_types, documents)

```Example: zip(['products', 'contracts'], ['Doc1 text', 'Doc2 text']) → [('products','Doc1 text'), ('contracts','Doc2 text')]```

The result is a list of hover texts, one for each point

```.update_layout()``` customaize the appearance 

```dict(r=20, b=10, l=10, t=40)``` → pixels for right, bottom, left, top margins.

In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()